[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C37_MLOps_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯标准库 + numpy/pandas、CPU 可跑、无需联网**，从零实现一套 ML 生命周期工具，每个工具都有 `assert` 自检。

这个 notebook 做三件事：① 确认环境；② 用一个最小例子体会「**为什么训练完才是开始**」（同一份代码，数据一漂移，模型就静默变差）；③ 立下全课的纪律——**每个工具都用 `assert` 验证它的不变量**。

## 1 · 环境自检

只需要 `numpy` 与 `pandas`。标准库 `hashlib`/`json`/`sqlite3`/`tempfile` 全自带，无需安装。

In [ ]:
import sys, platform, hashlib, json, sqlite3, tempfile
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
import pandas as pd
print('numpy', np.__version__, '| pandas', pd.__version__)
print('标准库 hashlib/json/sqlite3/tempfile 均可用')
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 亲眼看「训练完才是开始」：模型会静默劣化

训练一个最简单的阈值分类器（一维 logistic），在**训练分布**上评测得到一个准确率。

然后让世界**漂移**——输入分布整体平移（covariate shift）——再用同一个模型评测。模型没动一行代码、没报任何错，准确率却会掉下来。这就是 MLOps 要解决的根本问题。

In [ ]:
rng = np.random.default_rng(0)

def make_data(n, shift=0.0):
    # 两类一维高斯：负类均值 -1+shift，正类均值 +1+shift（标签由真实边界 x=shift 决定）
    y = rng.integers(0, 2, size=n)
    x = np.where(y == 1, 1.0, -1.0) + shift + rng.normal(0, 0.8, size=n)
    return x, y

def fit_threshold(x, y):
    # 训练：最优阈值取两类样本均值的中点（对等方差高斯就是贝叶斯最优边界）
    return 0.5 * (x[y == 1].mean() + x[y == 0].mean())

def accuracy(thr, x, y):
    pred = (x > thr).astype(int)
    return float((pred == y).mean())

# 在训练分布上训练并评测
x_tr, y_tr = make_data(4000, shift=0.0)
thr = fit_threshold(x_tr, y_tr)
acc_train_dist = accuracy(thr, *make_data(4000, shift=0.0))
print(f'学到的阈值 thr = {thr:+.3f}')
print(f'训练分布上的准确率   = {acc_train_dist:.3f}')
assert acc_train_dist > 0.85, '在训练分布上应当不错'

In [ ]:
# 世界漂移了：输入整体右移（covariate shift），模型却没更新阈值
for shift in [0.0, 0.5, 1.0, 1.5]:
    acc = accuracy(thr, *make_data(8000, shift=shift))
    flag = '  <- 还行' if acc > 0.8 else '  <- 已劣化！'
    print(f'分布平移 shift={shift:.1f}  准确率={acc:.3f}{flag}')

acc_drifted = accuracy(thr, *make_data(8000, shift=1.5))
assert acc_drifted < acc_train_dist - 0.05, '分布漂移后准确率应明显下降'
print('\n教训：模型没报错、代码没变，准确率却静默下滑。')
print('没有监控(模块04)，你根本不会知道；这就是为什么训练完才是开始。')

**关键结论**：传统软件部署后行为冻结，坏了会抛异常；ML 系统部署后行为随数据漂移，**坏了往往不报错、只是慢慢变差**。

本课五个模块就是给这个「活的、会变质的」系统装上：可复现资产（01 追踪 + 02 版本）、安全放行闸门（03 门禁）、上线后的体温计（04 监控/漂移）、随环境进化的能力（05 反馈/重训）。

## 3 · 立纪律：每个工具都验证它的「不变量」

本课每写一个工具，都会用 `assert` 钉死它必须满足的性质（invariant）。先热身两个贯穿全课的不变量。

**不变量 A：内容寻址是确定的**——相同内容必得相同哈希，内容变一比特哈希必变。这是模块 02 版本系统的地基。

In [ ]:
def content_hash(obj):
    '''对任意可 JSON 序列化对象做内容寻址哈希（排序键保证确定性）。'''
    blob = json.dumps(obj, sort_keys=True, ensure_ascii=False).encode('utf-8')
    return hashlib.sha256(blob).hexdigest()

a = {'lr': 0.01, 'layers': 3, 'data': 'v1'}
b = {'layers': 3, 'data': 'v1', 'lr': 0.01}   # 同内容，键序不同
c_changed = {'lr': 0.02, 'layers': 3, 'data': 'v1'}
assert content_hash(a) == content_hash(b), '相同内容必得相同哈希（与键序无关）'
assert content_hash(a) != content_hash(c_changed), '内容变一点哈希必变'
print('内容哈希(a) =', content_hash(a)[:16], '...')
print('✅ 不变量 A 成立：内容寻址确定且对改动敏感')

**不变量 B：可复现**——固定随机种子 + 固定配置，必得字节级相同的结果。不记录 seed 的实验几乎不可复现，这是模块 01 的核心。

In [ ]:
def train_run(seed, n=500):
    r = np.random.default_rng(seed)        # 种子完全决定随机流
    w = r.normal(size=4)
    return content_hash(np.round(w, 8).tolist())

assert train_run(42) == train_run(42), '同 seed 必复现'
assert train_run(42) != train_run(43), '不同 seed 结果不同'
print('seed=42 两次结果哈希一致：', train_run(42)[:16], '...')
print('✅ 不变量 B 成立：seed + 配置锁定可复现性')

## 4 · 一个会贯穿全课的小工具：临时隔离的工作目录

后面几个模块要往磁盘写真实文件（JSON 记录、内容寻址的 blob、sqlite 库）。用 `tempfile` 开一个**隔离的临时目录**，既是真实文件 I/O（不是假的内存模拟），又不会污染你的工作区。这也是真实工具做集成测试的标准做法。

In [ ]:
import os

def demo_real_file_io():
    with tempfile.TemporaryDirectory() as d:
        path = os.path.join(d, 'run_001.json')
        record = {'run_id': 'r001', 'params': {'lr': 0.01}, 'metrics': {'acc': 0.91}}
        with open(path, 'w', encoding='utf-8') as f:        # 真写盘
            json.dump(record, f, ensure_ascii=False)
        with open(path, encoding='utf-8') as f:             # 真读盘
            loaded = json.load(f)
        assert loaded == record, '写进去再读出来必须一致'
        return loaded['metrics']['acc']

acc = demo_real_file_io()
print(f'从临时文件读回的 acc = {acc}')
print('✅ 真实文件 I/O 跑通；临时目录已自动清理，工作区干净')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你写的每个工具（追踪器、版本库、门禁、漂移检测器、重训触发器）都会用 `assert` 钉死它的不变量；不变量成立则工具正确，工具正确则你彻底理解了对应 MLOps 平台在替你做什么。

**接下来五个模块**：01 实验追踪 → 02 数据与模型版本 → 03 评测门禁与 CI/CD → 04 监控与漂移 → 05 反馈闭环与重训练。每一步都建立在前一步之上，最后绕回起点形成闭环。

下一站：**模块 01 · 实验追踪** —— 这个模型是怎么来的？换个 seed 能复现吗？哪个 run 最好？